In [1]:
import gzip
import io
import re
from bs4 import BeautifulSoup
import numpy as np
import pandas as pd
import requests
import os

In [ ]:


baseURL = "https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/"

# Define standard columns upfront to optimize CSV parsing
TARGET_COLUMNS = [
    "BEGIN_YEARMONTH",
    "BEGIN_DAY",
    "BEGIN_TIME",
    "END_YEARMONTH",
    "END_DAY",
    "END_TIME",
    "EPISODE_ID",
    "EVENT_ID",
    "STATE",
    "STATE_FIPS",
    "YEAR",
    "MONTH_NAME",
    "EVENT_TYPE",
    "CZ_TYPE",
    "CZ_FIPS",
    "CZ_NAME",
    "WFO",
    "BEGIN_DATE_TIME",
    "CZ_TIMEZONE",
    "END_DATE_TIME",
    "INJURIES_DIRECT",
    "INJURIES_INDIRECT",
    "DEATHS_DIRECT",
    "DEATHS_INDIRECT",
    "DAMAGE_PROPERTY",
    "DAMAGE_CROPS",
    "SOURCE",
    "FLOOD_CAUSE",
    "BEGIN_RANGE",
    "BEGIN_AZIMUTH",
    "BEGIN_LOCATION",
    "END_RANGE",
    "END_AZIMUTH",
    "END_LOCATION",
    "BEGIN_LAT",
    "BEGIN_LON",
    "END_LAT",
    "END_LON",
    "EPISODE_NARRATIVE",
    "EVENT_NARRATIVE",
    "DATA_SOURCE",
]


def get_matching_file_urls(file_type="details", year=None):
    # Added timeout=30 to prevent server hang
    response = requests.get(baseURL, timeout=30)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")

    year_pattern = str(year) if year else r"\d{4}"
    pattern = (
        rf"StormEvents_{file_type}-ftp_v1\.0_d{year_pattern}_c.*\.csv\.gz$"
    )

    file_urls = []
    for link in soup.find_all("a", href=True):
        href = link["href"]
        if re.match(pattern, href):
            file_urls.append(baseURL + href)
    return file_urls


def load_storm_data_to_df(
    file_url, state="IOWA", event_types=None, month=None, wfo=None
):
    if event_types is None:
        event_types = ["FLOOD", "FLASH FLOOD"]

    print(f"Fetching: {file_url}")
    # Added timeout=30 to prevent downloading stalls
    res = requests.get(file_url, timeout=30)
    res.raise_for_status()

    with gzip.GzipFile(fileobj=io.BytesIO(res.content)) as gz:
        # Load only necessary columns to minimize memory footprint
        available_cols = lambda col: col in TARGET_COLUMNS
        df = pd.read_csv(gz, low_memory=False, usecols=available_cols)

    df["BEGIN_DATE_TIME"] = pd.to_datetime(df["BEGIN_DATE_TIME"])
    df["END_DATE_TIME"] = pd.to_datetime(df["END_DATE_TIME"])

    # Base filters
    is_target_state = df["STATE"].astype(str).str.upper() == state.upper()
    is_target_event = (
        df["EVENT_TYPE"].astype(str).str.upper().isin(event_types)
    )

    # Mandatory filters
    condition = is_target_state & is_target_event

    # Optional month filter
    if month is not None:
        condition &= df["BEGIN_DATE_TIME"].dt.month == month

    # Optional WFO filter
    if wfo is not None:
        condition &= df["WFO"].astype(str).str.upper() == wfo.upper()

    filtered_df = df[condition].copy()

    if filtered_df.empty:
        return pd.DataFrame()

    # Process episodes with resetting 72-hour window and true event durations
    window_hours = 72
    processed_groups = []

    for ep_id, group in filtered_df.groupby("EPISODE_ID"):
        group = group.sort_values("BEGIN_DATE_TIME").copy()
        window_indices = []
        macro_starts = []

        current_window_idx = 0
        current_macro_start = None

        for dt in group["BEGIN_DATE_TIME"]:
            if current_macro_start is None:
                current_macro_start = dt
            elif (
                dt - current_macro_start
            ).total_seconds() / 3600.0 >= window_hours:
                current_window_idx += 1
                current_macro_start = dt

            window_indices.append(current_window_idx)
            macro_starts.append(current_macro_start)

        group["MACRO_START"] = macro_starts
        group["NEW_EPISODE_ID"] = (
            str(ep_id)
            + "_"
            + pd.Series(window_indices, index=group.index).astype(str)
        )

        # Calculate actual latest end time per sub-episode window
        sub_episode_max_end = group.groupby("NEW_EPISODE_ID")[
            "END_DATE_TIME"
        ].transform("max")
        max_allowed_end = group["MACRO_START"] + pd.Timedelta(
            hours=window_hours
        )

        # Cap at 72 hours max, but preserve actual shorter durations
        group["CAPPED_EPISODE_END"] = np.where(
            sub_episode_max_end > max_allowed_end,
            max_allowed_end,
            sub_episode_max_end,
        )
        group["CAPPED_EPISODE_END"] = pd.to_datetime(
            group["CAPPED_EPISODE_END"]
        )

        group["TOTAL_EPISODE_DURATION"] = (
            group["CAPPED_EPISODE_END"] - group["MACRO_START"]
        )
        processed_groups.append(group)

    if processed_groups:
        filtered_df = pd.concat(processed_groups, ignore_index=True)
    else:
        filtered_df = pd.DataFrame(columns=filtered_df.columns)

    return filtered_df


if __name__ == "__main__":
    start_year = 2021
    end_year = 2024

    all_years_data = []

    for yr in range(start_year, end_year + 1):
        print(f"\n[1/3] Scraping file URL for {yr}...")
        try:
            urls = get_matching_file_urls(file_type="details", year=yr)
            if urls:
                target_url = urls[-1]
                print(
                    f"[2/3] Downloading & Processing {yr}... ({target_url.split('/')[-1]})"
                )

                yearly_df = load_storm_data_to_df(
                    target_url,
                    state="IOWA",
                    event_types=["FLOOD", "FLASH FLOOD"],
                    month=None,
                    wfo=None,
                )
                if not yearly_df.empty:
                    all_years_data.append(yearly_df)
                print(f"[3/3] Done with {yr}! Found {len(yearly_df)} records.")
            else:
                print(f"No URL found for year {yr}")
        except requests.exceptions.Timeout:
            print(f"ERROR: Downloading year {yr} timed out.")
        except Exception as e:
            print(f"ERROR processing year {yr}: {e}")

    if all_years_data:
        ia_floods_multiyear = pd.concat(all_years_data, ignore_index=True)

        # Desired output order
        cols_to_show = [
            "MACRO_START",
            "CAPPED_EPISODE_END",
            "TOTAL_EPISODE_DURATION",
            "NEW_EPISODE_ID",
        ] + [c for c in TARGET_COLUMNS if c in ia_floods_multiyear.columns]

        display_cols = [
            c for c in cols_to_show if c in ia_floods_multiyear.columns
        ]
        ia_floods_multiyear = ia_floods_multiyear[display_cols]

        output_filename = "NOAA_Final_2021_2024.csv"
        ia_floods_multiyear.to_csv(output_filename, index=False)

        print(
            f"\nSUCCESS: Saved {len(ia_floods_multiyear)} records to '{output_filename}'"
        )
    else:
        print("No data retrieved for the specified years.")


[1/3] Scraping file URL for 2021...


In [3]:

# 1. Load your master NOAA Storm Events CSV file
# (Adjust file path to your actual NOAA dataset)
noaa_df = pd.read_csv("NOAA_Final_2021_2024.csv")

# 2. Create an output directory to store the individual episode CSVs
output_dir = "noaa_episodes"
os.makedirs(output_dir, exist_ok=True)

# 3. Group by EPISODE_ID and export each group into its own CSV
print("Splitting dataset by EPISODE_ID...")

for episode_id, group in noaa_df.groupby("EPISODE_ID"):
    # Define filename using the unique Episode ID
    filename = f"episode_{episode_id}.csv"
    file_path = os.path.join(output_dir, filename)

    # Export group dataframe to CSV without row indices
    group.to_csv(file_path, index=False)

print(
    f"Successfully exported {noaa_df['EPISODE_ID'].nunique()} episode CSV files to '{output_dir}/'."
)

Splitting dataset by EPISODE_ID...
Successfully exported 97 episode CSV files to 'noaa_episodes/'.


In [5]:
import pandas as pd

# 1. Load the dataset
df = pd.read_csv("noaa_with_huc08.csv")

# Ensure EPISODE_ID is treated as a string for clean user matching
df["EPISODE_ID"] = df["EPISODE_ID"].astype(str)
if "NEW_EPISODE_ID" in df.columns:
    df["NEW_EPISODE_ID"] = df["NEW_EPISODE_ID"].astype(str)

# 2. Ask user for input
user_input = input("Enter the EPISODE_ID (or NEW_EPISODE_ID) you want to look at: ").strip()

# 3. Filter dataset for the selected episode
# Matches either standard EPISODE_ID or capped NEW_EPISODE_ID
episode_df = df[(df["EPISODE_ID"] == user_input) | (df["NEW_EPISODE_ID"] == user_input)]

# 4. Process and return results
if episode_df.empty:
    print(f"\nNo events found for Episode ID: '{user_input}'. Please check the ID and try again.")
else:
    print(f"\n--- EPISODE SUMMARY FOR ID: {user_input} ---")
    
    # Print general narrative if present
    if "EPISODE_NARRATIVE" in episode_df.columns and pd.notnull(episode_df["EPISODE_NARRATIVE"].iloc[0]):
        print(f"\nNarrative: {episode_df['EPISODE_NARRATIVE'].iloc[0]}\n")
    
    # Extract unique HUC-8 watersheds
    watersheds = episode_df[["HUC8", "NAME"]].drop_duplicates().dropna()
    
    print("=" * 50)
    print(f"INTERSECTING HUC-8 WATERSHEDS ({len(watersheds)} total):")
    print("=" * 50)
    for idx, row in watersheds.iterrows():
        print(f" • HUC-8: {int(row['HUC8'])} | Name: {row['NAME']}")
        
    print("\n" + "=" * 50)
    print(f"INDIVIDUAL EVENTS IN THIS EPISODE ({len(episode_df)} total):")
    print("=" * 50)
    
    event_summary = episode_df[["EVENT_ID", "EVENT_TYPE", "CZ_NAME", "BEGIN_DATE_TIME", "HUC8", "NAME"]].rename(
        columns={"NAME": "WATERSHED_NAME"}
    )
    print(event_summary.to_string(index=False))


--- EPISODE SUMMARY FOR ID: 193311_0 ---

Narrative: Heavy rain fell over portions of northern Iowa, especially northwestern Iowa and bordering states later on June 20 through early on June 21. This rainfall caused flash flooding in portions of northwest Iowa and also aided in saturating the soil. As another round of heavy rain fell later on June 21 into the night and morning of June 22, this rainfall led to renewed flash flooding. This rainfall made its way into streams and rivers resulting in significant and record river flooding in northwest Iowa including the West Fork of the Des Moines River. Flooding resulted in massive sandbagging efforts, prolonged road closures and damages, and impacts to home and businesses.||Note: damage estimates are taken from Iowa Homeland Security and Emergency Management Department and represent total flood damages for the entire county for the month of June. Damages are estimated to have surpassed $12 million in June.

INTERSECTING HUC-8 WATERSHEDS (5